In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import sys

sys.path.append("..")

import torch
import numpy as np
import matplotlib.pyplot as plt

## Data visualization

In [ ]:
filename2 = "K15"

In [ ]:
# set working dir
directory = "/system/user/publicdata/gyrokinetics/cyclone4_2"
# Input file names
filename = "Poten00000200"

path = os.path.join(directory, filename)

# load data files
time = np.loadtxt(os.path.join(directory, "time.dat"))
sgrid = np.loadtxt(os.path.join(directory, "sgrid"))
xphi = np.loadtxt(os.path.join(directory, "xphi"))
yphi = np.loadtxt(os.path.join(directory, "yphi"))
fluxes = np.loadtxt(os.path.join(directory, "fluxes.dat"))
kyspec = np.loadtxt(os.path.join(directory, "kyspec"))
krho = np.loadtxt(os.path.join(directory, "krho"))
vpgr = np.loadtxt(os.path.join(directory, "vpgr.dat"))
vperp = np.loadtxt(os.path.join(directory, "vperp.dat"))

In [ ]:
# number of parallel direction grid points
ns = sgrid.shape[1] if len(sgrid.shape) > 1 else sgrid.shape[0]

# number of x, y grid points (in real space)
nx, ny = xphi.shape[1], xphi.shape[0]

# number of modes in x and y direction
nkx, nky = krho.shape[1], krho.shape[0]

# get velocity space resolutions
nvpar, nmu = vpgr.shape[1], vpgr.shape[0]

# load real space electric potential data
a = np.loadtxt(os.path.join(directory, filename))
phi = np.reshape(a, (nx, ns, ny))

# Plot real space potential
plt.figure()
plt.pcolor(xphi, yphi, np.squeeze(phi[:, 8, :]).T, shading="auto")
plt.colorbar()
plt.title("Real Space Potential")
plt.xlabel("xphi")
plt.ylabel("yphi")
plt.show()

# Plot flux trace
plt.figure()
plt.plot(time, fluxes[:, 1])
plt.title("Flux Trace")
plt.xlabel("Time")
plt.ylabel("Fluxes")
plt.show()

# Load full distribution function data
with open(os.path.join(directory, filename2), "rb") as fid:
    ff = np.fromfile(fid, dtype=np.float64)

# Reshape the distribution function (first index is re/im component of fourier weights)
f = np.reshape(ff, (2, nvpar, nmu, ns, nkx, nky), order="F")

# Plot distribution function in velocity space
plt.figure()
plt.pcolor(vpgr, vperp, np.squeeze(f[0, :, :, 8, 80, 20]).T, shading="auto")
plt.colorbar()
plt.title("Distribution Function (Velocity Space)")
plt.xlabel("vpgr")
plt.ylabel("vperp")
plt.show()

# Additional plot of the distribution function
plt.figure()
plt.pcolor(np.squeeze(f[0, :, 2, :, 80, 20]).T, shading="auto")
plt.colorbar()
plt.title("Distribution Function Slice")
plt.show()

## Model visualization

In [ ]:
CKP = "/system/user/publicdata/gyrokinetics_checkpoints/cyclone4_2_swin_ckp"
device = "cuda"

In [ ]:
import yaml
from omegaconf import OmegaConf

from utils import load_model_and_config
from models import get_model

cfg = OmegaConf.create(yaml.safe_load(open(f"{CKP}/config.yaml", "r")))

model = get_model(cfg)

model, _, _ = load_model_and_config(f"{CKP}/ckp.pth", model, device)

model = model.to(device)

In [ ]:
from dataset.cyclone import CycloneDataset

data = CycloneDataset(
    in_memory=False, normalize=True, split="val", test_ratio=0.0, random_seed=cfg.seed
)
traindata = CycloneDataset(
    in_memory=False, normalize=True, split="train", val_ratio=0.0, test_ratio=0.0, random_seed=cfg.seed
)
print(f"Train: {len(traindata)}, Val: {len(data)}")

In [ ]:
from tqdm import tqdm
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt


def mse_time_histogram(model_fn, data):
    model_fn.eval()

    losses = defaultdict(list)
    sample = data[0]
    K1 = sample.x.numpy()
    x = torch.tensor(K1).to(device).unsqueeze(0)
    with torch.no_grad():
        for idx in tqdm(range(len(data))):
            sample = data[idx]
            K2 = sample.y.numpy()
            # x = torch.tensor(K1).to(device).unsqueeze(0)
            ts = sample.timestep_index.to(device).unsqueeze(0)
            x = model_fn(x, timestep=ts)

            mse = np.mean((x.squeeze(0).cpu().detach().numpy() - K2) ** 2)
            losses[ts.squeeze().item()].append(mse)
    
    fig = mse_time_histogram_from_losses(losses)
    
    return fig, losses

def mse_time_histogram_from_losses(losses):
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    times = sorted(losses.keys())
    losses_mean = [np.mean(losses[t]) for t in times]
    losses_std = [np.std(losses[t]) for t in times]

    # Bar plot with error bars
    ax.bar(times, losses_mean, yerr=losses_std, alpha=0.7, capsize=5, color="blue")
    ax.set_xlabel("Time Step")
    ax.set_ylabel("Mean Squared Error")
    ax.set_title("MSE by Time Step")
    ax.grid(True)

    return fig

In [ ]:
_, losses = mse_time_histogram(model, data=traindata)

In [ ]:
_, val_losses = mse_time_histogram(model, data=data)

In [ ]:
_ = mse_time_histogram_from_losses(losses | val_losses)

In [ ]:
from matplotlib import colormaps


def force_aspect(ax, aspect=1):
    im = ax.get_images()
    extent = im[0].get_extent()
    ax.set_aspect(abs((extent[1] - extent[0]) / (extent[3] - extent[2])) / aspect)


def distribution_5D(model_fn, idx=0):
    labels = ["vpar", "mu", "s", "x", "y"]

    model_fn.eval()
    sample = traindata[idx]
    K2 = sample.y.numpy()
    x = sample.x.to(device).unsqueeze(0)
    ts = sample.timestep_index.to(device).unsqueeze(0)
    f = model_fn(x, timestep=ts).squeeze(0).cpu().detach().numpy()

    comb = torch.combinations(torch.arange(5), 2).tolist()

    fig, ax = plt.subplots(5, 5, figsize=(20, 20))
    c_map = colormaps["rainbow"]
    c_map.set_bad("k")

    imin = -1
    for i, j in comb:
        other = tuple([o for o in range(5) if o != i and o != j])
        xx = f[0].std(other)
        xx[xx == 0] = np.nan
        ax[i, j].matshow(xx, cmap=c_map)

        if i > imin:
            ax[i, j].set_ylabel(labels[i], fontsize=20)
            ax[i, j].set_xlabel(labels[j], fontsize=20)
            imin = i

        force_aspect(ax[i, j])

    for i in range(5):
        for j in range(5):
            if [i, j] not in comb:
                ax[i, j].remove()
    return fig

In [ ]:
fig = distribution_5D(model, 10)
# fig

In [ ]:
fig

In [ ]:
from matplotlib import animation


def animate_5D(model_fn, title="", frames=10, start=55, idx=0):
    plt.rcParams["animation.html"] = "jshtml"
    plt.ioff()
    plt.gca().set_aspect("equal")

    fig, ax = plt.subplots(2, 1, figsize=(10, 10))
    fig.tight_layout()
    fig.suptitle(title)

    model_fn.eval()
    sample = traindata[idx]
    K1 = sample.x.numpy()
    K2 = sample.y.numpy()
    x = torch.tensor(K1).to(device).unsqueeze(0)
    f = model_fn(x).squeeze(0).cpu().detach().numpy()

    gt_vmax, gt_vmin = K2[0, :, 7, :, 85, :].max(), K2[0, :, 7, :, 85, :].min()
    pred_vmax, pred_vmin = f[0, :, 7, :, 85, :].max(), f[0, :, 7, :, 85, :].min()

    # Initial plots to set up the colorbars
    gt_im = ax[0].matshow(K2[0, start, 7, :, 85, :], vmax=gt_vmax, vmin=gt_vmin)
    pred_im = ax[1].matshow(f[0, start, 7, :, 85, :], vmax=pred_vmax, vmin=pred_vmin)

    # Adding colorbars
    cbar_gt = fig.colorbar(
        gt_im, ax=ax[0], orientation="vertical", fraction=0.046, pad=0.04
    )
    cbar_gt.set_label("Ground Truth", fontsize=12)

    cbar_pred = fig.colorbar(
        pred_im, ax=ax[1], orientation="vertical", fraction=0.046, pad=0.04
    )
    cbar_pred.set_label("Prediction", fontsize=12)

    def animate(t):
        gt3 = K2[0, start + t, 7, :, 85, :]
        pred3 = f[0, start + t, 7, :, 85, :]

        gt_im.set_array(gt3)
        pred_im.set_array(pred3)

    return animation.FuncAnimation(fig, animate, frames=frames)

In [ ]:
ani = animate_5D(model, "", frames=32, start=0, idx=0)
ani

In [ ]:
writer = animation.PillowWriter(fps=8, bitrate=400)
ani.save("shift2.gif", writer=writer)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation


def animate_5D2(model_fn, title="", frames=10, start=55, idx=0):
    labels = ["vpar", "mu", "s", "x", "y"]

    plt.rcParams["animation.html"] = "jshtml"
    plt.ioff()
    plt.gca().set_aspect("equal")

    fig, ax = plt.subplots(2, 5, figsize=(15, 10))
    fig.tight_layout()
    fig.suptitle(title)

    model_fn.eval()
    sample = data[idx]
    K1 = sample.x.numpy()
    K2 = sample.y.numpy()
    x = torch.tensor(K1).to(device).unsqueeze(0)
    f = model_fn(x).squeeze(0).cpu().detach().numpy()

    # Compute limits for each subplot
    gt_limits = {
        "vpar": (K2[:, :, 0, :, 85, :].max(), K2[:, :, 0, :, 85, :].min()),
        "mu": (K2[:, 0, :, :, 85, :].max(), K2[:, 0, :, :, 85, :].min()),
        "s": (K2[:, :, :, :, 85, :].max(), K2[:, :, :, :, 85, :].min()),
        "x": (K2[:, :, :, :, 85, :].max(), K2[:, :, :, :, 85, :].min()),
        "y": (K2[:, :, :, :, :, :].max(), K2[:, :, :, :, :, :].min()),
    }
    pred_limits = {
        "vpar": (f[:, :, 0, :, 85, :].max(), f[:, :, 0, :, 85, :].min()),
        "mu": (f[:, 0, :, :, 85, :].max(), f[:, 0, :, :, 85, :].min()),
        "s": (f[:, :, :, :, 85, :].max(), f[:, :, :, :, 85, :].min()),
        "x": (f[:, :, :, :, 85, :].max(), f[:, :, :, :, 85, :].min()),
        "y": (f[:, :, :, :, :, :].max(), f[:, :, :, :, :, :].min()),
    }

    # Initialize matshow and colorbars
    ims = []
    cbars = []
    for row in range(2):
        for col in range(5):
            label = labels[col]
            vmin, vmax = gt_limits[label] if row == 0 else pred_limits[label]
            im = ax[row, col].matshow(
                K2[0, 0, 0, :, 85, :] if row == 0 else f[0, 0, 0, :, 85, :],
                vmin=vmin,
                vmax=vmax,
            )
            ims.append(im)
            cbar = fig.colorbar(
                im, ax=ax[row, col], orientation="vertical", fraction=0.046, pad=0.04
            )
            cbar.set_label(
                f"{label} ({'Ground Truth' if row == 0 else 'Prediction'})", fontsize=12
            )
            cbars.append(cbar)

            # Set labels
            ax[row, col].set_xlabel(label, fontsize=15)
            ax[row, col].set_ylabel("s" if col == 2 else "y", fontsize=15)

    def animate(t):
        if start + t < K2.shape[2]:
            ims[0].set_array(K2[0, :, 0, start + t, 85, :])
            ims[5].set_array(f[0, :, 0, start + t, 85, :])

        if start + t < K2.shape[3]:
            ims[1].set_array(K2[0, 0, :, start + t, 85, :])
            ims[6].set_array(f[0, 0, :, start + t, 85, :])

        if start + t < K2.shape[0]:
            ims[2].set_array(K2[0, start + t, 0, :, 85, :])
            ims[7].set_array(f[0, start + t, 0, :, 85, :])

        if start + t < K2.shape[-1]:
            ims[3].set_array(K2[0, :, 0, :, 85, start + t])
            ims[8].set_array(f[0, :, 0, :, 85, start + t])

        ims[4].set_array(K2[0, 0, 0, :, start + t, :])
        ims[9].set_array(f[0, 0, 0, :, start + t, :])

    return animation.FuncAnimation(fig, animate, frames=frames)

In [ ]:
ani = animate_5D2(model, "", frames=5, start=0)
ani

In [ ]:
writer = animation.PillowWriter(fps=4, bitrate=400)
ani.save("big_ani.gif", writer=writer)

In [ ]:
def force_aspect(ax, aspect=1):
    im = ax.get_images()
    extent = im[0].get_extent()
    ax.set_aspect(abs((extent[1] - extent[0]) / (extent[3] - extent[2])) / aspect)


def plot4x4(f, title="", mark_bad=False):
    labels = ["vpar", "mu", "s", "x", "y"]
    comb = torch.combinations(torch.arange(5), 2).tolist()

    fig, ax = plt.subplots(5, 5, figsize=(20, 20))
    fig.suptitle(title)
    c_map = colormaps["coolwarm"]
    c_map.set_bad("k")

    for i, j in comb:
        other = tuple([o for o in range(5) if o != i and o != j])
        xx = f[0].mean(other)
        if mark_bad:
            xx_std = f[0].std(other)
            xx[xx_std == 0] = np.nan

        im00 = ax[i, j].imshow(xx.T, cmap=c_map)

        fig.colorbar(im00, ax=ax[i, j])
        ax[i, j].set_xlabel(labels[i])
        ax[i, j].set_ylabel(labels[j])
        force_aspect(ax[i, j])

    for i in range(5):
        for j in range(5):
            if [i, j] not in comb:
                ax[i, j].remove()
    return fig

In [ ]:
model.eval()
K1, _, K2, _ = data[10]
K1 = K1.numpy()
K2 = K2.numpy()
x = torch.tensor(K1).to(device).unsqueeze(0)
f = model(x).squeeze(0).cpu().detach().numpy()

# f = np.zeros_like(f)

mse = K2 - f

plot4x4(K2, "GT")

In [ ]:
plot4x4(f, f"PRED (MSE = {mse.mean():.4f})")

In [ ]:
plot4x4(mse, f"MSE = {mse.mean():.4f}")

In [ ]:
def force_aspect(ax, aspect=1):
    im = ax.get_images()
    extent = im[0].get_extent()
    ax.set_aspect(abs((extent[1] - extent[0]) / (extent[3] - extent[2])) / aspect)


def plot4x4_sided(x1, x2, title="", mark_bad=False):
    labels = ["vpar", "mu", "s", "x", "y"]
    comb = torch.combinations(torch.arange(5), 2).tolist()

    fig, ax = plt.subplots(5, 5, figsize=(30, 12))
    for i in range(5):
        for j in range(5):
            ax[i, j].remove()

    fig.tight_layout()
    fig.suptitle(title)
    c_map = colormaps["coolwarm"]
    c_map.set_bad("k")

    for i, j in comb:
        other = tuple([o for o in range(5) if o != i and o != j])
        x1_plot = x1[0].mean(other)
        x2_plot = x2[0].mean(other)
        x1_vmax, x1_vmin = x1_plot.max(), x1_plot.min()
        x2_vmax, x2_vmin = x2_plot.max(), x2_plot.min()

        if mark_bad:
            x1_std = x1.std(other)
            x2_std = x2.std(other)
            x1_plot[x1_std == 0] = np.nan
            x2_plot[x2_std == 0] = np.nan

        # Clear the axis and directly plot two images side by side
        ax_ij = ax[i, j]
        # ax_ij.clear()

        # Get the position of the original axis
        pos = ax_ij.get_position()

        # Create two new axes within the same space as the original subplot
        width = pos.width / 2  # Split the width into two halves
        ax1 = fig.add_axes([pos.x0, pos.y0, width, pos.height])
        ax2 = fig.add_axes([pos.x0 + width, pos.y0, width, pos.height])

        # Plot x1 and xp side by side
        im1 = ax1.imshow(x1_plot, cmap=c_map, vmax=x1_vmax, vmin=x1_vmin)
        im2 = ax2.imshow(x2_plot, cmap=c_map, vmax=x2_vmax, vmin=x2_vmin)

        fig.colorbar(im1, ax=ax1)
        fig.colorbar(im2, ax=ax2)

        if i == 0:
            # Set axis labels
            ax1.set_title("GT")
            ax2.set_title("PRED")

        ax1.set_xlabel(labels[i])
        ax1.set_ylabel(labels[j])
        ax2.set_xlabel(labels[i])
        ax2.set_ylabel(labels[j])

        # Force aspect ratio
        force_aspect(ax1)
        force_aspect(ax2)

    return fig

In [ ]:
model.eval()
sample = data[0]
K1 = sample.x.numpy()
K2 = sample.y.numpy()
x = torch.tensor(K1).to(device).unsqueeze(0)
ts = sample.timestep_index.to(device).unsqueeze(0)
f = model.cpu()(x.cpu(), timestep=ts.cpu()).squeeze(0).cpu().detach().numpy()

fig = plot4x4_sided(K1, K2)

In [ ]:
fig

In [ ]:
model.eval()
sample = data[1]
K1 = sample.x.numpy()
K2 = sample.y.numpy()
x = torch.tensor(K1).to(device).unsqueeze(0)
ts = sample.timestep_index.to(device).unsqueeze(0)
f = model.cpu()(x.cpu(), timestep=ts.cpu()).squeeze(0).cpu().detach().numpy()

fig = plot4x4_sided(K1, K2)

In [ ]:
model.eval()
sample = data[2]
K1 = sample.x.numpy()
K2 = sample.y.numpy()
x = torch.tensor(K1).to(device).unsqueeze(0)
ts = sample.timestep_index.to(device).unsqueeze(0)
f = model.cpu()(x.cpu(), timestep=ts.cpu()).squeeze(0).cpu().detach().numpy()

fig = plot4x4_sided(K1, K2)

In [ ]:
from matplotlib import animation


def force_aspect(ax, aspect=1):
    im = ax.get_images()
    extent = im[0].get_extent()
    ax.set_aspect(abs((extent[1] - extent[0]) / (extent[3] - extent[2])) / aspect)


def plot4x4_animate(model_fn, title="", mark_bad=False, frames=5, start_idx=0):
    plt.rcParams["animation.html"] = "jshtml"
    plt.ioff()

    labels = ["vpar", "mu", "s", "x", "y"]
    comb = torch.combinations(torch.arange(5), 2).tolist()

    fig, ax = plt.subplots(5, 5, figsize=(30, 12))
    for i in range(5):
        for j in range(5):
            ax[i, j].remove()

    # fig.tight_layout()
    fig.suptitle(title)
    c_map = colormaps["coolwarm"]
    c_map.set_bad("k")
    
    model.eval()
    
    x0 = data[start_idx].x
    preds = []
    preds = [x0.to(device).unsqueeze(0)]
    with torch.no_grad():
        for i in range(frames):
            xp = model_fn(preds[-1])
            preds.append(xp)

    preds = preds[1:]
    preds = [p.squeeze(0).cpu().detach().numpy() for p in preds]

    def animate(t):
        x1 = data[start_idx + t].y.numpy()
        xp = preds[t]
        ts = data[start_idx + t].timestep.numpy().item()
        fig.suptitle(f"ts={ts:.2f}", fontsize=30)

        for i, j in comb:
            other = tuple([o for o in range(5) if o != i and o != j])

            x1_plot = x1[0].mean(other)
            xp_plot = xp[0].mean(other)
            x1_vmax, x1_vmin = x1_plot.max(), x1_plot.min()
            xp_vmax, xp_vmin = xp_plot.max(), xp_plot.min()

            if mark_bad:
                x1_std = x1.std(other)
                xp_std = xp.std(other)
                x1_plot[x1_std == 0] = np.nan
                xp_plot[xp_std == 0] = np.nan

            # Clear the axis and directly plot two images side by side
            ax_ij = ax[i, j]
            # ax_ij.clear()

            # Get the position of the original axis
            pos = ax_ij.get_position()

            # Create two new axes within the same space as the original subplot
            width = pos.width / 2  # Split the width into two halves
            ax1 = fig.add_axes([pos.x0, pos.y0, width, pos.height])
            ax2 = fig.add_axes([pos.x0 + width, pos.y0, width, pos.height])

            # Plot x1 and xp side by side
            im1 = ax1.imshow(x1_plot, cmap=c_map, vmax=x1_vmax, vmin=x1_vmin)
            im2 = ax2.imshow(xp_plot, cmap=c_map, vmax=xp_vmax, vmin=xp_vmin)

            if i == 0:
                # Set axis labels
                ax1.set_title("GT")
                ax2.set_title("PRED")
            ax1.set_xlabel(labels[i])
            ax1.set_ylabel(labels[j])
            ax2.set_xlabel(labels[i])
            ax2.set_ylabel(labels[j])

            # Force aspect ratio
            force_aspect(ax1)
            force_aspect(ax2)

    return animation.FuncAnimation(fig, animate, frames=frames)

In [ ]:
ani = plot4x4_animate(model, frames=10, start_idx=0)
ani

In [ ]:
writer = animation.PillowWriter(fps=1, bitrate=600)
ani.save("sided.gif", writer=writer)